# CLAMPfull cross-fitted proportion recovery

A single train/test split gives one unstable Pearson correlation per cell type, especially for datasets with only a few dozen held-out samples. This notebook instead runs donor-grouped 5-fold cross-validation (repeated 80/20 holdouts as a fallback for datasets with too few donor groups), with a fixed, pre-specified LV budget per dataset (reused from the full-dataset model) so a train/test split never collapses model capacity to a handful of LVs.

For each fold and each cell type $c$:

1. Fit CLAMPfull on the training samples only, with the dataset's fixed LV count.
2. Select $k_c^* = \arg\max_k\; \operatorname{cor}(B_{\mathrm{train},k}, p_{\mathrm{train},c})$ using training samples only.
3. Fit a linear calibration on training samples only: $p_{\mathrm{train},c} = \alpha_c + \beta_c B_{\mathrm{train},k_c^*}$.
4. Project the held-out samples into the trained model and predict $\hat p_{\mathrm{test},c} = \alpha_c + \beta_c B_{\mathrm{test},k_c^*}$, using the fixed LV and calibration (no reselection, no refitting on test).

Every sample ends up with exactly one out-of-fold (OOF) prediction per cell type. Pooling OOF predictions per cell type, we report:

- **OOF Pearson** — does the LV track relative abundance?
- **OOF R²** (vs. the pooled OOF mean-proportion baseline) — does it predict fractions better than guessing the mean?
- **OOF MAE** — how far are predicted fractions from true fractions?
- **LV selection stability** — since each fold is an independently fit CLAMPfull model, raw LV index isn't comparable across folds. Stability is instead measured by correlating the selected LV's gene-loading vector against the reference fold's, and reporting the fraction of folds above a similarity threshold.

## Parameters

In [ ]:
OUT_DIR <- 'output/03_model_biology/02_pseudobulk/01_holdout80'
DATASETS <- 'Brain_Mathys2023,Brain_Xiong2023,Heart_Datar2026,PBMC_1k1k,PBMC_Perez2022,Lung_Sikkema2023'
DATA_ROOT <- 'data/pseudobulk'
MODEL_ROOT <- 'output/01_model_building/05_pseudobulk'
SEED <- 42L
MAX_ITER <- 500L
N_FOLDS <- 5L
MIN_UNITS_FOR_KFOLD <- 2L * N_FOLDS
FALLBACK_REPEATS_SMALL <- 50L
FALLBACK_REPEATS_MODERATE <- 20L
STABILITY_THRESHOLD <- 0.7

## Functions

In [ ]:
suppressPackageStartupMessages({
  library(CLAMP); library(rsvd); library(data.table); library(here)
  library(ggplot2); library(dplyr)
})
source(here('scripts', 'pseudobulk', 'common.R'))

prepare_train_test <- function(train_counts, test_counts,
                               mean_cutoff = 0.5, var_cutoff = 0.1) {
  train_cpm <- CLAMP::cpmCLAMP(train_counts)
  prep <- CLAMP::preprocessCLAMP(train_cpm, mean_cutoff, var_cutoff)
  train_norm <- CLAMP::zscoreCLAMP(prep$Y_filtered, prep$rowStats)

  test_cpm <- CLAMP::cpmCLAMP(test_counts)
  genes <- intersect(rownames(prep$Y_filtered), rownames(test_cpm))
  test_norm <- CLAMP::zscoreCLAMP(test_cpm[genes, , drop = FALSE],
                                  prep$rowStats[genes, , drop = FALSE])
  list(train = train_norm[genes, , drop = FALSE],
       test = test_norm, row_stats = prep$rowStats[genes, , drop = FALSE])
}

# Fits CLAMPfull with a caller-supplied, fixed LV budget instead of a
# per-fold elbow -- keeps model capacity constant across folds so a
# smaller training split never collapses a dataset to a handful of LVs.
fit_clampfull_fixed_k <- function(norm, fixed_k, max_iter, prior_mat_full) {
  cap <- min(dim(norm)) - 1L
  if (fixed_k > cap) {
    stop(sprintf(
      'fixed_k (%d) exceeds this fold\'s capacity (%d); dim(norm) = %s',
      fixed_k, cap, paste(dim(norm), collapse = 'x')))
  }
  # rsvd must compute at least fixed_k singular vectors -- CLAMPbase/CLAMPfull
  # index svdres$d[clamp_k] without bounds-checking on an externally supplied
  # svdres, so svd_k < fixed_k would silently produce NA and corrupt the fit.
  svd_k_fold <- min(max(fixed_k, floor((min(dim(norm)) - 1L) / 4L)), cap)
  svdres <- rsvd::rsvd(norm, k = svd_k_fold)
  prior <- CLAMP::getMatchedPathwayMat(prior_mat_full, rownames(norm))
  base <- CLAMP::CLAMPbase(Y = norm, svdres = svdres, clamp_k = fixed_k, trace = FALSE)
  full <- CLAMP::CLAMPfull(Y = norm, svdres = svdres, priorMat = prior,
    clamp.base.result = base, clamp_k = fixed_k, max.iter = max_iter,
    use_cpp = TRUE, trace = FALSE)
  full$B <- as.data.frame(full$B); colnames(full$B) <- colnames(norm)
  full$Z <- as.data.frame(full$Z); rownames(full$Z) <- rownames(norm)
  full
}

select_training_lvs <- function(B, truth) {
  samples <- intersect(rownames(B), rownames(truth))
  B <- B[samples, , drop = FALSE]
  truth <- truth[samples, , drop = FALSE]
  B <- B[, apply(B, 2, var, na.rm = TRUE) > 0, drop = FALSE]
  scores <- cor(B, truth, use = 'pairwise.complete.obs', method = 'pearson')
  rows <- lapply(colnames(scores), function(ct) {
    values <- scores[, ct]
    lv <- names(which.max(values))
    data.frame(cell_type = ct, LV = lv, train_cor = unname(values[lv]),
               stringsAsFactors = FALSE)
  })
  do.call(rbind, rows)
}

# Linear calibration mapping a selected LV's raw B-values to true
# proportions, fit on training samples only.
fit_calibration <- function(b_train, p_train) {
  fit <- lm(p_train ~ b_train)
  list(alpha = unname(coef(fit)[1]), beta = unname(coef(fit)[2]))
}

predict_calibration <- function(alpha, beta, b_test) alpha + beta * b_test

make_split <- function(ds, ids, data_root, seed) {
  resolved <- resolve_split_groups(ds, ids, data_root)
  group <- resolved$group
  units <- sort(unique(unname(group)))
  if (length(units) < 2L) stop(ds, ': fewer than two split units')
  set.seed(seed)
  n_train <- min(length(units) - 1L, max(1L, floor(0.8 * length(units))))
  train_units <- sample(units, n_train, replace = FALSE)
  train_ids <- sort(ids[group[ids] %in% train_units])
  test_ids <- sort(setdiff(ids, train_ids))
  membership <- data.frame(dataset = ds, sample = ids,
    split_unit = resolved$split_unit, group_id = unname(group[ids]),
    split = ifelse(ids %in% train_ids, 'train', 'test'), seed = seed,
    stringsAsFactors = FALSE)
  list(train = train_ids, test = test_ids, membership = membership)
}

# Raw LV index isn't comparable across folds: each fold is an independent
# CLAMPfull fit with no cross-fold anchoring, so "LV5" in two folds are
# unrelated axes. Stability is instead the correlation between the
# selected LV's gene-loading vector (Z column) in a reference fold and in
# every other fold, on the shared gene intersection.
lv_stability_pairs <- function(assignment_list, Z_list, reference_fold, threshold) {
  ref <- assignment_list[[as.character(reference_fold)]]
  other_folds <- setdiff(names(assignment_list), as.character(reference_fold))
  rows <- lapply(other_folds, function(fold) {
    cmp <- assignment_list[[fold]]
    common_ct <- intersect(ref$cell_type, cmp$cell_type)
    do.call(rbind, lapply(common_ct, function(ct) {
      lv_ref <- ref$LV[ref$cell_type == ct]
      lv_cmp <- cmp$LV[cmp$cell_type == ct]
      genes <- intersect(rownames(Z_list[[as.character(reference_fold)]]),
                          rownames(Z_list[[fold]]))
      r <- suppressWarnings(cor(
        Z_list[[as.character(reference_fold)]][genes, lv_ref],
        Z_list[[fold]][genes, lv_cmp], use = 'pairwise.complete.obs'))
      data.frame(cell_type = ct, reference_fold = reference_fold,
        compare_fold = as.integer(fold), n_common_genes = length(genes),
        loading_cor = r, is_stable = !is.na(r) && abs(r) > threshold,
        stringsAsFactors = FALSE)
    }))
  })
  do.call(rbind, rows)
}

# Pooled out-of-fold Pearson, R^2 (vs. the pooled OOF mean-proportion
# baseline) and MAE per (dataset, cell_type).
compute_oof_metrics <- function(oof_dt) {
  oof_dt[, .(
    oof_pearson = suppressWarnings(cor(predicted_fraction, true_fraction,
                                        use = 'complete.obs')),
    oof_r2 = 1 - sum((true_fraction - predicted_fraction)^2, na.rm = TRUE) /
                 sum((true_fraction - mean(true_fraction, na.rm = TRUE))^2, na.rm = TRUE),
    oof_mae = mean(abs(true_fraction - predicted_fraction), na.rm = TRUE),
    n_oof_samples = sum(!is.na(predicted_fraction)),
    n_folds = uniqueN(fold_id)
  ), by = .(dataset, cell_type)]
}

## Analysis and outputs

In [ ]:
out_dir <- OUT_DIR
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
datasets <- strsplit(DATASETS, ',', fixed = TRUE)[[1]]

# GO-BP prior is dataset/fold-independent -- fetch and build the sparse
# matrix once instead of redoing it inside every one of the ~30 fold fits.
pathways <- CLAMP::getGMT(
  'https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025')
names(pathways) <- paste0('BP_', names(pathways))
prior_mat_full <- CLAMP::gmtListToSparseMat(list(BP = pathways))

splits <- list(); assignments <- list(); oof_rows <- list(); stability_rows <- list()

for (ds in datasets) {
  message('Running cross-fitted validation: ', ds)
  counts <- read_csv_matrix(file.path(DATA_ROOT, ds, 'bulk_expr.csv'))
  truth <- read_truth(file.path(DATA_ROOT, ds, 'truthFrac_v0.csv'))
  ids <- sort(intersect(colnames(counts), rownames(truth)))
  fixed_k <- as.integer(data.table::fread(file.path(MODEL_ROOT, ds, 'k.csv'))$k[1])

  split <- make_kfold_splits(ds, ids, DATA_ROOT, k = N_FOLDS,
    min_units_for_kfold = MIN_UNITS_FOR_KFOLD,
    fallback_repeats_small = FALLBACK_REPEATS_SMALL,
    fallback_repeats_moderate = FALLBACK_REPEATS_MODERATE, seed = SEED)

  assignment_list <- list(); z_list <- list()

  for (fold_info in split$folds) {
    fold_id <- fold_info$fold_id
    train_ids <- fold_info$train; test_ids <- fold_info$test

    splits[[length(splits) + 1L]] <- data.frame(
      dataset = ds, sample = ids, split_unit = split$split_unit,
      group_id = unname(split$group[ids]), fold_id = fold_id,
      split_mode = split$mode,
      split = ifelse(ids %in% test_ids, 'test', 'train'), seed = SEED,
      stringsAsFactors = FALSE)

    prepared <- prepare_train_test(counts[, train_ids, drop = FALSE],
                                   counts[, test_ids, drop = FALSE])
    model <- fit_clampfull_fixed_k(prepared$train, fixed_k, MAX_ITER, prior_mat_full)
    train_B <- t(as.matrix(model$B))
    assignment <- select_training_lvs(train_B,
      truth[rownames(train_B), , drop = FALSE])

    calib <- lapply(seq_len(nrow(assignment)), function(i) {
      fit_calibration(train_B[, assignment$LV[i]], truth[rownames(train_B), assignment$cell_type[i]])
    })
    assignment$alpha <- vapply(calib, function(x) x$alpha, numeric(1))
    assignment$beta <- vapply(calib, function(x) x$beta, numeric(1))
    assignment$dataset <- ds; assignment$fold_id <- fold_id
    assignment$fixed_k <- fixed_k; assignment$n_train <- length(train_ids)
    assignments[[length(assignments) + 1L]] <- assignment
    assignment_list[[as.character(fold_id)]] <- assignment
    z_list[[as.character(fold_id)]] <- as.matrix(model$Z)

    model$Z <- as.matrix(model$Z[rownames(prepared$test), , drop = FALSE])
    projected <- CLAMP::projectCLAMP(CLAMPres = model, newdata = prepared$test)
    test_B <- t(as.matrix(projected))
    test_B <- test_B[test_ids, , drop = FALSE]

    for (i in seq_len(nrow(assignment))) {
      ct <- assignment$cell_type[i]; lv <- assignment$LV[i]
      raw_score <- test_B[test_ids, lv]
      predicted <- predict_calibration(assignment$alpha[i], assignment$beta[i], raw_score)
      oof_rows[[length(oof_rows) + 1L]] <- data.frame(
        dataset = ds, cell_type = ct, sample = test_ids, fold_id = fold_id,
        LV = lv, raw_lv_score = raw_score,
        true_fraction = truth[test_ids, ct], predicted_fraction = predicted,
        seed = SEED, stringsAsFactors = FALSE)
    }
  }

  if (length(assignment_list) >= 2L) {
    reference_fold <- as.integer(names(assignment_list)[1])
    stab <- lv_stability_pairs(assignment_list, z_list, reference_fold, STABILITY_THRESHOLD)
    stab$dataset <- ds
    stability_rows[[length(stability_rows) + 1L]] <- stab
  }
}

split_df <- rbindlist(splits)
assignment_df <- rbindlist(assignments)
oof_df <- rbindlist(oof_rows)
stability_df <- rbindlist(stability_rows)

oof_metrics <- compute_oof_metrics(oof_df)
oof_summary <- oof_metrics[, .(
  n_cell_types = .N,
  median_oof_pearson = median(oof_pearson, na.rm = TRUE),
  mean_oof_pearson = mean(oof_pearson, na.rm = TRUE)), by = dataset]
stability_summary <- stability_df[, .(
  stability_rate = mean(is_stable, na.rm = TRUE),
  mean_abs_loading_cor = mean(abs(loading_cor), na.rm = TRUE),
  n_fold_comparisons = .N), by = .(dataset, cell_type)]

fwrite(split_df, file.path(out_dir, 'split_membership.csv'))
fwrite(assignment_df, file.path(out_dir, 'train_lv_assignments.csv'))
fwrite(oof_df, file.path(out_dir, 'oof_predictions.csv'))
fwrite(oof_metrics, file.path(out_dir, 'oof_metrics.csv'))
fwrite(oof_summary, file.path(out_dir, 'oof_summary.csv'))
fwrite(stability_df, file.path(out_dir, 'lv_stability.csv'))
fwrite(stability_summary, file.path(out_dir, 'lv_stability_summary.csv'))

# Figure 1 (per dataset): true vs. cross-fitted predicted proportion, one
# panel per cell type, annotated with pooled OOF Pearson/R2/MAE.
for (ds in datasets) {
  df_ds <- oof_df[dataset == ds]
  met_ds <- oof_metrics[dataset == ds]
  ann <- merge(met_ds, df_ds[, .(x = min(true_fraction, na.rm = TRUE),
                                  y = max(predicted_fraction, na.rm = TRUE)), by = cell_type],
               by = 'cell_type')
  ann[, label := sprintf('r=%.2f  R2=%.2f  MAE=%.3f', oof_pearson, oof_r2, oof_mae)]

  p_scatter <- ggplot(df_ds, aes(true_fraction, predicted_fraction)) +
    geom_abline(slope = 1, intercept = 0, color = 'grey70', linetype = 'dashed') +
    geom_point(alpha = 0.6, size = 1.4) +
    geom_text(data = ann, aes(x = x, y = y, label = label),
              hjust = 0, vjust = 1, size = 2.6, inherit.aes = FALSE) +
    facet_wrap(~ cell_type, scales = 'free') +
    theme_bw(base_size = 10) +
    labs(x = 'True held-out proportion', y = 'Cross-fitted predicted proportion',
         title = paste0(ds, ': out-of-fold proportion recovery'))
  ggsave(file.path(out_dir, paste0('oof_scatter_', ds, '.png')), p_scatter,
         width = 10, height = 7, dpi = 150)
}

# Figure 2: distribution of pooled OOF Pearson correlation per cell type, by dataset.
p_oof <- ggplot(oof_metrics, aes(x = reorder(dataset, oof_pearson, median),
                                  y = oof_pearson)) +
  geom_hline(yintercept = 0, color = 'grey75') +
  geom_boxplot(outlier.shape = NA, width = 0.55) +
  geom_jitter(width = 0.14, alpha = 0.65, size = 1.6) +
  coord_flip() + theme_bw(base_size = 11) +
  labs(x = NULL, y = 'Out-of-fold Pearson correlation',
       title = 'CLAMPfull cross-fitted validation',
       subtitle = 'Fixed LV budget per dataset; LV and calibration selected on training folds only')
ggsave(file.path(out_dir, 'oof_pearson_boxplot.png'), p_oof, width = 8, height = 5.5, dpi = 150)
print(p_oof)
oof_summary